# Eyewear brand localization / OCR

This notebook runs the standalone `eyewear-localization/` pipeline. It first tests OCR independently, then optionally enables class-agnostic SAM3 eyewear localization. Brand names are never sent to SAM3.

## 1. Clone and install

The OCR models are installed in a UV environment. EasyOCR downloads its model weights on first use.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/fez-Ox/pxModel-Object-Counting.git"
REPO_DIR = Path("/kaggle/working/pxModel-localization")
APP_DIR = REPO_DIR / "eyewear-localization"

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

subprocess.run(["uv", "sync", "--extra", "ocr"], cwd=APP_DIR, check=True)
print("Pipeline:", APP_DIR)


## 2. Select an image

Edit `IMAGE_PATH` if the automatic first-image selection is not the image you want.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidates = sorted(
    path for path in Path("/kaggle/input").rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
IMAGE_PATH = candidates[0] if candidates else Path("/kaggle/input/your-dataset/image.jpg")  # edit me
BRAND_FILE = APP_DIR / "brands.txt"  # edit or replace with your catalog
OUTPUT_DIR = APP_DIR / "outputs" / "notebook"
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Image:", IMAGE_PATH)
print("Device:", DEVICE)
assert IMAGE_PATH.exists(), f"Update IMAGE_PATH: {IMAGE_PATH}"


## 3. OCR-only test

This stage does not need SAM3. It detects all text, matches the configured brand gazetteer, and writes `text_detections[]` and `signs[]`. A sign is not assigned to an eyewear instance at this stage.

In [ ]:
def run_pipeline(checkpoint=None):
    command = [
        "uv", "run", "python", "infer.py", str(IMAGE_PATH),
        "--brand-file", str(BRAND_FILE),
        "--ocr-backend", "easyocr",
        "--device", DEVICE,
        "--no-vlm-audit",
        "--out", str(OUTPUT_DIR),
    ]
    if checkpoint is not None:
        command += ["--sam3-checkpoint", str(checkpoint)]
    completed = subprocess.run(command, cwd=APP_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        completed.check_returncode()
    result_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}.json"
    return json.loads(result_path.read_text())

ocr_result = run_pipeline()
print("OCR text detections:")
for detection in ocr_result["text_detections"]:
    print(f"  {detection['text']!r} ({detection['confidence']:.2f})")
print("Gazetteer-matched signs:")
for sign in ocr_result["signs"]:
    print(f"  {sign['text']!r} -> {sign['brand']}")


In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open(OUTPUT_DIR / f"{IMAGE_PATH.stem}.jpg"))


## 4. Optional full attribution test with SAM3

Attach an approved `sam3.pt` as a Kaggle dataset, then edit the path below. The SAM3 checkpoint is gated and is not downloaded automatically.

In [ ]:
SAM3_CHECKPOINT = Path("/kaggle/input/your-sam3-dataset/sam3.pt")  # edit me

if SAM3_CHECKPOINT.exists():
    full_result = run_pipeline(SAM3_CHECKPOINT)
    print("Per-instance decisions:")
    for output in full_result["outputs"]:
        print(output["instance_id"], output["brand"], output["probabilities"])
    display(Image.open(OUTPUT_DIR / f"{IMAGE_PATH.stem}.jpg"))
else:
    print("Attach a permitted SAM3 dataset and update SAM3_CHECKPOINT to run the full pipeline.")


## 5. Inspect the JSON contract

The result keeps raw OCR, scoped signs, cue evidence, and final decisions separate for auditing.

In [ ]:
from pprint import pprint
pprint({
    "signs": ocr_result["signs"],
    "evidence": ocr_result["evidence"],
    "outputs": ocr_result["outputs"],
})
